In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Cell 2 — Install dependencies
%pip install anthropic deepeval jsonlines tqdm pandas tiktoken sentence-transformers -q


In [ ]:
# Cell 3 — Load config + (optional) API key
import pathlib
import sys

p = pathlib.Path.cwd().resolve()
while p != p.parent and not (p / "synthetic_data").exists():
    p = p.parent
sys.path.append(str(p))

from synthetic_data.colab.config import *

try:
    from google.colab import userdata
    import anthropic

    _key = userdata.get("ANTHROPIC_API_KEY")
    client = anthropic.Anthropic(api_key=_key) if _key else None
except Exception:
    client = None

print("Setup complete ✓")


In [ ]:
# Cell 4 — Checkpoint helper
import json
import os

import jsonlines


def save_checkpoint(data, filename):
    path = f"{SYNTHETIC_DIR}/{filename}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with jsonlines.open(path, "w") as w:
        w.write_all(data)
    print(f"Saved {len(data)} items -> {path} ✓")


def load_checkpoint(filename):
    path = f"{SYNTHETIC_DIR}/{filename}"
    if not os.path.exists(path):
        return []
    with jsonlines.open(path) as r:
        return list(r)


In [ ]:
import glob
import os
from collections import Counter

from synthetic_data.pipeline.filtering import filter_chunks, load_all_chunks

# ── RUN ────────────────────────────────────────────────────────
all_chunks = load_all_chunks(CHUNKS_DIR)
print(f"Loaded {len(all_chunks)} total chunks")

kept_chunks, rejected_chunks = filter_chunks(all_chunks, min_tokens=MIN_TOKENS, max_tokens=MAX_TOKENS)
print(f"Kept: {len(kept_chunks)} | Rejected: {len(rejected_chunks)}")

save_checkpoint(kept_chunks, "filtered_chunks.jsonl")
save_checkpoint(rejected_chunks, "rejected/rejected_chunks.jsonl")

# ── SUMMARY REPORT ─────────────────────────────────────────────
tier_counts = Counter(c.get("tier") for c in kept_chunks)
print("\nFiltered chunk distribution:")
for tier, count in sorted(tier_counts.items()):
    print(f"  {tier}: {count} chunks")
